**Imports**

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt


**Load CIFAR‑10**

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train_full = x_train_full / 255.0
x_test       = x_test / 255.0


**Create 3 splits: train / validation / test**

In [ ]:
x_train = x_train_full[:40000]
y_train = y_train_full[:40000]

x_val   = x_train_full[40000:]
y_val   = y_train_full[40000:]


**Create tf.data pipelines**

In [ ]:
batch_size = 64

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(5000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


**Build a simple model that overfits easily**

In [ ]:
def create_simple_model():
    model = tf.keras.Sequential([
        # Block 1
        tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
        tf.keras.layers.MaxPooling2D(),

        # Block 2 (NEW)
        tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


model_no_aug = create_simple_model()
model_no_aug.summary()


**Train the model (few epochs, still overfits)**

In [ ]:
epochs = 20

history_no_aug = model_no_aug.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)


**Check train and test accuracy**

In [ ]:
train_loss, train_acc = model_no_aug.evaluate(x_train, y_train, verbose=0)
print("Train Accuracy:", train_acc)

test_loss, test_acc = model_no_aug.evaluate(x_test, y_test, verbose=0)
print("Test Accuracy:", test_acc)


**Data Augmentation Block**

In [ ]:
# This Sequential block creates NEW random variations of each image
# every time the model sees it during training.
# This prevents memorization and reduces overfitting.

data_augmentation = tf.keras.Sequential([

    # Randomly flip images left ↔ right.
    # Helps the model learn that orientation does not change the class.
    tf.keras.layers.RandomFlip("horizontal"),

    # Rotate images by up to ±10% of a full turn (≈ ±36 degrees).
    # Makes the model robust to tilted objects.
    tf.keras.layers.RandomRotation(0.1),

    # Randomly zoom in or out by 10%.
    # Helps the model handle objects at different scales.
    tf.keras.layers.RandomZoom(0.1),

    # Randomly change image contrast by ±10%.
    # Makes the model robust to lighting differences.
    tf.keras.layers.RandomContrast(0.1)

], name="data_augmentation")


**Apply Augmentation to Training Data**

In [ ]:
# We create a new training dataset that applies augmentation
# ONLY to the training images — never to validation or test.
# This ensures the model learns robust features but is evaluated fairly.

train_ds_aug = tf.data.Dataset.from_tensor_slices((x_train, y_train))

train_ds_aug = (
    train_ds_aug
    # Shuffle the training data so batches are always different.
    .shuffle(5000)

    # Group images into batches of 64 for efficient training.
    .batch(64)

    # Apply augmentation to each batch.
    # 'training=True' ensures layers like RandomFlip behave randomly.
    .map(lambda x, y: (data_augmentation(x, training=True), y),
         num_parallel_calls=tf.data.AUTOTUNE)

    # Prefetch prepares the next batch while the GPU is training.
    # This speeds up training significantly.
    .prefetch(tf.data.AUTOTUNE)
)


**Validation and Test Sets**

In [ ]:
# Validation set: NO augmentation, because we want a clean evaluation
# of how well the model generalizes.
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)

# Test set: also NO augmentation.
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)


**Train the Model WITH Augmentation**

In [ ]:
# We create a NEW model so the comparison is fair.
# Same architecture as before — only the data changes.
model_aug = create_simple_model()

# Train the model using the augmented training dataset.
# Notice: validation data is NOT augmented.
history_aug = model_aug.fit(
    train_ds_aug,      # augmented training data
    validation_data=val_ds,   # clean validation data
    epochs=20,          # same number of epochs as before
    verbose=1          # show progress
)


**Evaluate Train and Test Accuracy**

In [ ]:
# Evaluate on the augmented training data
train_loss_aug, train_acc_aug = model_aug.evaluate(x_train, y_train, verbose=0)
print("Train Accuracy (with augmentation):", train_acc_aug)

# Evaluate on the clean test set
test_loss_aug, test_acc_aug = model_aug.evaluate(x_test, y_test, verbose=0)
print("Test Accuracy (with augmentation):", test_acc_aug)
